In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# Physical and numerical parameters
# ==========================================================

Lx = 200.0
Ly = 200.0

Nx = 100
Ny = 100

dx = Lx / Nx
dy = Ly / Ny

x = np.linspace(0, Lx, Nx)
y = np.linspace(0, Ly, Ny)
X, Y = np.meshgrid(x, y)

# Wind velocity
vx = 2.3          # m/s
vy = 0.5          # m/s

# Turbulent diffusion
Dx = 1.0          # m^2/s
Dy = 1.0          # m^2/s

# Surface deposition
lam = 0.005       # 1/s

# Mixing height
H = 50.0          # m

# Particle settling velocity
Vs = 0.186        # m/s

# Source
Q = 100.0
xf = 100.0
yf = 100.0
sigma = 2.5       # m

# Time step
dt = 0.25         # s

# Convergence criterion
tolerance = 1.0   # percent

# Selected times for convergence testing
test_times = np.array([15, 30, 45, 60, 90, 120, 150, 180, 210, 240], dtype=float)

print(f"dx = {dx:.3f} m")
print(f"dy = {dy:.3f} m")
print(f"dt = {dt:.3f} s")
print(f"Settling velocity Vs = {Vs:.3f} m/s")
print(f"Convergence tolerance = {tolerance:.2f}%")

# ==========================================================
# Stability diagnostics
# ==========================================================

Cx = abs(vx) * dt / dx
Cy = abs(vy) * dt / dy

rx = Dx * dt / dx**2
ry = Dy * dt / dy**2

combined_number = Cx + Cy + 2.0 * (rx + ry)

print("==============================================")
print("NUMERICAL STABILITY DIAGNOSTICS")
print("==============================================")
print(f"Cx = {Cx:.4f}")
print(f"Cy = {Cy:.4f}")
print(f"rx = {rx:.4f}")
print(f"ry = {ry:.4f}")
print(f"Cx + Cy + 2(rx + ry) = {combined_number:.4f}")
print("==============================================")
print("The diagnostic value is reported for transparency.")
print("The exact stability requirement should be stated")
print("consistently with the numerical scheme used in the manuscript.")

# ==========================================================
# Gaussian source
# ==========================================================

Source = Q * np.exp(
    -((X - xf)**2 + (Y - yf)**2) / (2 * sigma**2)
)

# Initial concentration
u = np.zeros((Ny, Nx), dtype=float)

# Combined first-order removal coefficient
removal_rate = lam + Vs / H

print(f"Settling term Vs/H = {Vs/H:.6f} 1/s")
print(f"Total removal rate = lambda + Vs/H = {removal_rate:.6f} 1/s")
print(f"Characteristic combined removal time = {1/removal_rate:.2f} s")

# ==========================================================
# One explicit finite-difference time step
# ==========================================================

def advance_one_step(u):
    un = u.copy()

    # Interior nodes
    dudx = (un[1:-1, 1:-1] - un[1:-1, :-2]) / dx
    dudy = (un[1:-1, 1:-1] - un[:-2, 1:-1]) / dy

    d2udx2 = (
        un[1:-1, 2:]
        - 2.0 * un[1:-1, 1:-1]
        + un[1:-1, :-2]
    ) / dx**2

    d2udy2 = (
        un[2:, 1:-1]
        - 2.0 * un[1:-1, 1:-1]
        + un[:-2, 1:-1]
    ) / dy**2

    reaction = -removal_rate * un[1:-1, 1:-1]

    u[1:-1, 1:-1] = (
        un[1:-1, 1:-1]
        + dt * (
            -vx * dudx
            -vy * dudy
            +Dx * d2udx2
            +Dy * d2udy2
            +reaction
            +Source[1:-1, 1:-1]
        )
    )

    # Zero-concentration boundary conditions
    u[:, 0] = 0.0
    u[:, -1] = 0.0
    u[0, :] = 0.0
    u[-1, :] = 0.0

    return u

# ==========================================================
# Run simulation and store selected concentration fields
# ==========================================================

max_time = float(np.max(test_times))
max_steps = int(round(max_time / dt))

# Ensure exact requested output times are represented by integer steps
requested_steps = {int(round(t / dt)): float(t) for t in test_times}

u = np.zeros((Ny, Nx), dtype=float)
snapshots = {}

for step in range(1, max_steps + 1):
    u = advance_one_step(u)

    if step in requested_steps:
        snapshots[requested_steps[step]] = u.copy()

print(f"Simulation completed to {max_time:.2f} s.")
print(f"Total time steps = {max_steps}")
print(f"Stored snapshots = {len(snapshots)}")

# ==========================================================
# Calculate convergence metrics
# ==========================================================

times = np.array(sorted(snapshots.keys()), dtype=float)

max_concentration = np.array([
    np.max(snapshots[t]) for t in times
])

hotspot_indices = [
    np.unravel_index(np.argmax(snapshots[t]), snapshots[t].shape)
    for t in times
]

hotspot_x = np.array([x[i[1]] for i in hotspot_indices])
hotspot_y = np.array([y[i[0]] for i in hotspot_indices])

relative_changes = np.full(len(times), np.nan)

for k in range(1, len(times)):
    current = snapshots[times[k]]
    previous = snapshots[times[k - 1]]

    numerator = np.linalg.norm(current - previous)
    denominator = np.linalg.norm(current)

    relative_changes[k] = 100.0 * numerator / denominator

print("==========================================================")
print("TEMPORAL CONVERGENCE RESULTS")
print("==========================================================")
print(f"{'Time (s)':>10} {'Max C':>15} {'Hotspot x':>12} {'Hotspot y':>12} {'Change (%)':>15}")
print("----------------------------------------------------------")

for k, t in enumerate(times):
    change_text = "---" if np.isnan(relative_changes[k]) else f"{relative_changes[k]:.6f}"
    print(
        f"{t:10.2f} "
        f"{max_concentration[k]:15.6f} "
        f"{hotspot_x[k]:12.2f} "
        f"{hotspot_y[k]:12.2f} "
        f"{change_text:>15}"
    )

# ==========================================================
# Determine the convergence time
# ==========================================================

converged_time = None

for k in range(1, len(times)):
    if relative_changes[k] < tolerance:
        converged_time = times[k]
        break

print("==========================================================")
print("CONVERGENCE DECISION")
print("==========================================================")

if converged_time is not None:
    print(
        f"First selected time satisfying the "
        f"{tolerance:.1f}% criterion: {converged_time:.2f} s"
    )
else:
    print(
        f"The {tolerance:.1f}% criterion was not reached "
        "within the tested time range."
    )

# Specifically inspect 120 s
if 120.0 in snapshots:
    idx120 = np.where(times == 120.0)[0][0]
    print(f"\nMaximum concentration at 120 s: {max_concentration[idx120]:.6f}")
    print(
        f"Hotspot at 120 s: "
        f"({hotspot_x[idx120]:.2f}, {hotspot_y[idx120]:.2f}) m"
    )
    if not np.isnan(relative_changes[idx120]):
        print(
            f"Relative field change at 120 s: "
            f"{relative_changes[idx120]:.8f}%"
        )

# ==========================================================
# Relative field change versus time
# ==========================================================

plt.figure(figsize=(8, 5))

plt.semilogy(
    times[1:],
    relative_changes[1:],
    marker='o',
    linewidth=2
)

plt.axhline(
    tolerance,
    linestyle='--',
    linewidth=1.5,
    label=f'{tolerance:.0f}% criterion'
)

if converged_time is not None:
    plt.axvline(
        converged_time,
        linestyle=':',
        linewidth=1.5,
        label=f'First convergence: {converged_time:.0f} s'
    )

plt.xlabel("Simulation time (s)")
plt.ylabel("Relative field change (%)")
plt.title("Temporal Convergence of Airborne Microplastic Concentration")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# ==========================================================
# Maximum concentration versus time
# ==========================================================

plt.figure(figsize=(8, 5))

plt.plot(
    times,
    max_concentration,
    marker='o',
    linewidth=2
)

plt.xlabel("Simulation time (s)")
plt.ylabel("Maximum concentration")
plt.title("Temporal Evolution of Maximum Airborne Microplastic Concentration")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ==========================================================
# Hotspot location versus time
# ==========================================================

plt.figure(figsize=(8, 5))

plt.plot(
    times,
    hotspot_x,
    marker='o',
    linewidth=2,
    label='Hotspot x-coordinate'
)

plt.plot(
    times,
    hotspot_y,
    marker='s',
    linewidth=2,
    label='Hotspot y-coordinate'
)

plt.xlabel("Simulation time (s)")
plt.ylabel("Coordinate (m)")
plt.title("Temporal Stability of Hotspot Location")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# ==========================================================
# Spatial concentration fields
# ==========================================================

display_times = [30.0, 60.0, 90.0, 120.0]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

vmax = max(np.max(snapshots[t]) for t in display_times)

for ax, t in zip(axes.ravel(), display_times):
    im = ax.contourf(
        X,
        Y,
        snapshots[t],
        levels=50,
        cmap='plasma',
        vmin=0,
        vmax=vmax
    )

    ax.scatter(
        xf,
        yf,
        c='white',
        s=70,
        edgecolors='black',
        label='Emission source'
    )

    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title(f"Airborne Microplastic Concentration at t = {t:.0f} s")

fig.colorbar(
    im,
    ax=axes.ravel().tolist(),
    label="Microplastic concentration"
)

plt.tight_layout()
plt.show()

# ==========================================================
# Final assessment of 120 s
# ==========================================================

idx120 = np.where(times == 120.0)[0]

if len(idx120) == 0:
    print("120 s was not included in the stored test times.")
else:
    k = idx120[0]

    print("==========================================================")
    print("ASSESSMENT OF t = 120 s")
    print("==========================================================")
    print(f"Maximum concentration : {max_concentration[k]:.6f}")
    print(
        f"Hotspot location      : "
        f"({hotspot_x[k]:.2f}, {hotspot_y[k]:.2f}) m"
    )

    if np.isnan(relative_changes[k]):
        print("Relative field change : unavailable")
    else:
        print(
            f"Relative field change : "
            f"{relative_changes[k]:.8f}%"
        )

        if relative_changes[k] < tolerance:
            print(
                f"\nRESULT: The field-change criterion "
                f"({tolerance:.1f}%) is satisfied at 120 s."
            )
        else:
            print(
                f"\nRESULT: The field-change criterion "
                f"({tolerance:.1f}%) is NOT satisfied at 120 s."
            )

    print("\nNote:")
    print("Use the convergence result together with hotspot stability")
    print("when selecting the final simulation time for the manuscript.")

